In [1]:
! pip install opendatasets



# New section

In [2]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/puneet6060/intel-image-classification")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: karanchawla11
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification


100%|██████████| 346M/346M [00:05<00:00, 69.8MB/s]


A) SETUP + DATA (Intel → train/test)
1) Confirm seg_train path (run)

In [3]:
!ls -lah /content/intel-image-classification/seg_train
!ls -lah /content/intel-image-classification/seg_train/seg_train | head

total 12K
drwxr-xr-x 3 root root 4.0K Mar 17 13:27 .
drwxr-xr-x 5 root root 4.0K Mar 17 13:27 ..
drwxr-xr-x 8 root root 4.0K Mar 17 13:27 seg_train
total 432K
drwxr-xr-x 8 root root 4.0K Mar 17 13:27 .
drwxr-xr-x 3 root root 4.0K Mar 17 13:27 ..
drwxr-xr-x 2 root root  68K Mar 17 13:27 buildings
drwxr-xr-x 2 root root  68K Mar 17 13:27 forest
drwxr-xr-x 2 root root  68K Mar 17 13:27 glacier
drwxr-xr-x 2 root root  68K Mar 17 13:27 mountain
drwxr-xr-x 2 root root  64K Mar 17 13:27 sea
drwxr-xr-x 2 root root  68K Mar 17 13:27 street


2) Create /content/dataset/train and /content/dataset/test (run)

Use a bigger split now (more stable training):

In [4]:
import os, random
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

SOURCE_DIR = "/content/intel-image-classification/seg_train/seg_train"
OUTPUT_DIR = "/content/dataset"
TRAIN_DIR = os.path.join(OUTPUT_DIR, "train")
TEST_DIR  = os.path.join(OUTPUT_DIR, "test")
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# Clean old pngs
for d in (TRAIN_DIR, TEST_DIR):
    for f in os.listdir(d):
        if f.lower().endswith(".png"):
            os.remove(os.path.join(d, f))

SEED = 42
random.seed(SEED)

N_TRAIN = 400
N_TEST  = 80
N_TOTAL = N_TRAIN + N_TEST

IMG_W, IMG_H = 320, 240

all_images = []
for root, _, files in os.walk(SOURCE_DIR):
    for f in files:
        if f.lower().endswith((".jpg",".jpeg",".png")):
            all_images.append(os.path.join(root, f))

print("Found images:", len(all_images))
random.shuffle(all_images)
selected = all_images[:N_TOTAL]
train_list = selected[:N_TRAIN]
test_list  = selected[N_TRAIN:]

def save_imgs(paths, out_dir, prefix):
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGB")
        img = img.resize((IMG_W, IMG_H), Image.BILINEAR)
        img.save(os.path.join(out_dir, f"{prefix}_{i:05d}.png"), format="PNG", optimize=True)

save_imgs(train_list, TRAIN_DIR, "train")
save_imgs(test_list, TEST_DIR, "test")

print("Train:", len(os.listdir(TRAIN_DIR)))
print("Test :", len(os.listdir(TEST_DIR)))

Found images: 14034
Train: 400
Test : 80


B) **PYTORCH** LOADING    
**3) Create DataLoaders (run)**

In [5]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

TRAIN_DIR = "/content/dataset/train"
TEST_DIR  = "/content/dataset/test"
IMG_W, IMG_H = 320, 240

transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor()
])

class IntelFolderDataset(Dataset):
    def __init__(self, folder):
        self.paths = sorted([
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith((".png",".jpg",".jpeg"))
        ])
        if not self.paths:
            raise ValueError(f"No images found in {folder}")
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return transform(img)

train_ds = IntelFolderDataset(TRAIN_DIR)
test_ds  = IntelFolderDataset(TEST_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

train_loader = DataLoader(
    train_ds, batch_size=16, shuffle=True,
    num_workers=2 if DEVICE=="cuda" else 0,
    pin_memory=(DEVICE=="cuda")
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

print("Train:", len(train_ds), "Test:", len(test_ds), "Device:", DEVICE)
x = next(iter(train_loader))
print("Batch:", x.shape)  # [16, 3, 240, 320]

Train: 400 Test: 80 Device: cuda
Batch: torch.Size([16, 3, 240, 320])


C) AUTOENCODER (LoRa-feasible latent size)

This is important: your earlier latent created 1500+ packets.
We fix that by using 3 downsamples so latent is 30×40, not 60×80.

4) Define model (run)

In [6]:
import torch.nn as nn

class LoRaAutoEncoder(nn.Module):
    def __init__(self, latent_ch=16):
        super().__init__()
        # 240x320 -> 120x160 -> 60x80 -> 30x40
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, latent_ch, 3, stride=1, padding=1),
        )
        # 30x40 -> 60x80 -> 120x160 -> 240x320
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_ch, 64, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 16, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 3, 3, padding=1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z

model = LoRaAutoEncoder(latent_ch=16).to(DEVICE)
print("Model on:", DEVICE)

Model on: cuda


In [7]:
x = next(iter(test_loader)).to(DEVICE)
with torch.no_grad():
    _, z = model(x)
print("z shape:", z.shape)  # expect [1, 16, 30, 40]

latent_bytes = z.shape[1]*z.shape[2]*z.shape[3]
packets_51 = latent_bytes / 51
print("latent bytes:", latent_bytes)
print("~packets per image (51B):", packets_51)

z shape: torch.Size([1, 16, 30, 40])
latent bytes: 19200
~packets per image (51B): 376.47058823529414


D) TRAIN
6) Train model (run)

In [8]:
import torch
import torch.optim as optim
from tqdm import tqdm

EPOCHS = 25
LR = 1e-3

loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS+1):
    model.train()
    total = 0.0
    for x in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        x = x.to(DEVICE, non_blocking=True)
        out, _ = model(x)
        loss = loss_fn(out, x)

        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)

    print(f"Epoch {epoch} train_loss: {total/len(train_ds):.6f}")

torch.save(model.state_dict(), "/content/lora_autoencoder.pt")
print("Saved: /content/lora_autoencoder.pt")

Epoch 1/25: 100%|██████████| 25/25 [00:03<00:00,  7.18it/s]


Epoch 1 train_loss: 0.066264


Epoch 2/25: 100%|██████████| 25/25 [00:03<00:00,  6.73it/s]


Epoch 2 train_loss: 0.043867


Epoch 3/25: 100%|██████████| 25/25 [00:01<00:00, 14.23it/s]


...

Epoch 23 train_loss: 0.006352


Epoch 24/25: 100%|██████████| 25/25 [00:02<00:00, 10.71it/s]


Epoch 24 train_loss: 0.006201


Epoch 25/25: 100%|██████████| 25/25 [00:02<00:00, 11.56it/s]

Epoch 25 train_loss: 0.006060
Saved: /content/lora_autoencoder.pt


E) EVALUATE CLEAN (no loss)
7) Evaluate SSIM/PSNR clean (run)

In [9]:
!pip -q install scikit-image
import numpy as np, time, os
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

os.makedirs("/content/samples_clean", exist_ok=True)

def chw_to_hwc_float01(t):
    return t.detach().cpu().clamp(0,1).numpy().transpose(1,2,0).astype(np.float32)

def float01_to_u8(img):
    return (np.clip(img,0,1)*255.0).round().astype(np.uint8)

ssims, psnrs = [], []
model.eval()

with torch.no_grad():
    for i, x in enumerate(test_loader):
        x = x.to(DEVICE)
        out, _ = model(x)

        orig = chw_to_hwc_float01(x.squeeze(0))
        rec  = chw_to_hwc_float01(out.squeeze(0))

        ssims.append(ssim(orig, rec, channel_axis=2, data_range=1.0))
        psnrs.append(psnr(orig, rec, data_range=1.0))

        if i < 5:
            side = np.concatenate([float01_to_u8(orig), float01_to_u8(rec)], axis=1)
            Image.fromarray(side).save(f"/content/samples_clean/{i:02d}.png")

print("CLEAN SSIM avg:", float(np.mean(ssims)))
print("CLEAN PSNR avg:", float(np.mean(psnrs)))
print("Saved:", "/content/samples_clean/")

CLEAN SSIM avg: 0.5921093225479126
CLEAN PSNR avg: 23.138859203699184
Saved: /content/samples_clean/


F) LoRa SIMULATION (packet loss)
8) Run LoRa packet-loss experiment (run)

In [10]:
import numpy as np
import torch
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from PIL import Image
import os

os.makedirs("/content/samples_lora_loss", exist_ok=True)

def quantize_uint8(z_torch):
    z = z_torch.detach().cpu().numpy().astype(np.float32).squeeze(0)
    zmin = float(z.min())
    zmax = float(z.max())
    if abs(zmax - zmin) < 1e-8:
        z_u8 = np.zeros(z.size, dtype=np.uint8)
    else:
        z_u8 = ((z - zmin) / (zmax - zmin) * 255.0).round().astype(np.uint8).reshape(-1)
    return z_u8, zmin, zmax, z.shape

def dequantize_from_uint8(z_u8, zmin, zmax, shape):
    if abs(zmax - zmin) < 1e-8:
        z = np.full(np.prod(shape), zmin, dtype=np.float32)
    else:
        z = z_u8.astype(np.float32) / 255.0
        z = z * (zmax - zmin) + zmin
    z = z.reshape(shape).astype(np.float32)
    return torch.from_numpy(z).unsqueeze(0)

def packetize_bytes(b, payload=51):
    return [b[i:i+payload].copy() for i in range(0, len(b), payload)]

def apply_loss(packets, loss_rate, seed):
    rng = np.random.default_rng(seed)
    return [None if rng.random() < loss_rate else p for p in packets]

def depacketize(packets_loss, total_len, payload=51, fill=0):
    out = np.full(total_len, fill, dtype=np.uint8)
    cursor = 0
    for p in packets_loss:
        rem = total_len - cursor
        if rem <= 0: break
        n = min(payload, rem)
        if p is not None:
            out[cursor:cursor+n] = p[:n]
        cursor += n
    return out

def chw_to_hwc_float01(t):
    return t.detach().cpu().clamp(0,1).numpy().transpose(1,2,0).astype(np.float32)

def float01_to_u8(img):
    return (np.clip(img,0,1)*255.0).round().astype(np.uint8)

PAYLOAD = 51
LOSS_RATES = [0.0, 0.10, 0.20, 0.30]

model.eval()
for lr in LOSS_RATES:
    ssims, psnrs = [], []

    with torch.no_grad():
        for idx, x in enumerate(test_loader):
            x = x.to(DEVICE)

            # encode
            _, z = model(x)

            # quantize + packetize
            z_u8, zmin, zmax, zshape = quantize_uint8(z)
            packets = packetize_bytes(z_u8, payload=PAYLOAD)

            # loss
            lost = apply_loss(packets, lr, seed=idx)

            # rebuild + dequantize
            rebuilt = depacketize(lost, total_len=len(z_u8), payload=PAYLOAD, fill=0)
            z_hat = dequantize_from_uint8(rebuilt, zmin, zmax, zshape).to(DEVICE)

            # decode
            out_hat = model.decoder(z_hat)

            # metrics
            orig = chw_to_hwc_float01(x.squeeze(0))
            rec  = chw_to_hwc_float01(out_hat.squeeze(0))
            ssims.append(ssim(orig, rec, channel_axis=2, data_range=1.0))
            psnrs.append(psnr(orig, rec, data_range=1.0))

            # save first sample
            if idx == 0:
                side = np.concatenate([float01_to_u8(orig), float01_to_u8(rec)], axis=1)
                Image.fromarray(side).save(f"/content/samples_lora_loss/loss_{int(lr*100):02d}.png")

    print(f"loss={lr:.2f} | SSIM={float(np.mean(ssims)):.3f} | PSNR={float(np.mean(psnrs)):.2f} dB | packets/img~{len(packets)}")

print("Saved: /content/samples_lora_loss/")

loss=0.00 | SSIM=0.592 | PSNR=23.14 dB | packets/img~377
loss=0.10 | SSIM=0.365 | PSNR=16.34 dB | packets/img~377
loss=0.20 | SSIM=0.284 | PSNR=13.18 dB | packets/img~377
loss=0.30 | SSIM=0.241 | PSNR=10.96 dB | packets/img~377
Saved: /content/samples_lora_loss/


In [14]:
from google.colab import files
files.download('/content/lora_autoencoder.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>